# 01. 전처리 — PDF → v5 표준화 parquet

## 이 노트북이 하는 일
1. 한국 LEED 건물 460개 PDF 스코어카드 로드
2. Rule 기반으로 v5 스키마로 표준화 (107개 매핑 규칙)
3. (선택) LLM 전문가 리뷰 — OPENAI_API_KEY 있을 때만
4. parquet 저장

## 출력
- `data/processed/project_features.parquet` (460행 × 28컬럼)
- `data/processed/standardized_credits.parquet` (9,747행)
- (LLM 옵션) `data/processed/project_features_option_a.parquet`

## 참고
- 자세한 설명: `docs/01_전처리_과정.md`
- 파이프라인 설계: `docs/02_파이프라인_설계.md`

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass

import os
import pandas as pd
from src.langgraph_workflow.graph import run_standardization
from src.data.loader import LEEDDataLoader

print(f'프로젝트 경로: {ROOT}')
print(f'OpenAI API 키: {"있음 (LLM 리뷰 가능)" if os.environ.get("OPENAI_API_KEY") else "없음 (Rule only)"}')

## 1단계: 데이터 로드

In [ ]:
csv_df = LEEDDataLoader().load_project_directory()
pdf_files = sorted((ROOT / 'data' / 'raw' / 'scorecards').glob('*.pdf'))
print(f'CSV 프로젝트: {len(csv_df)}개, PDF: {len(pdf_files)}개')

## 2단계: 전체 파이프라인 실행

각 PDF: PDF 파싱 → CSV 매칭 → Rule 매핑 → 수학 검증 → (LLM 리뷰) → 저장

**소요 시간**:
- API 키 없음 (Rule only): 약 5분
- API 키 있음 (LLM 리뷰): 약 8시간 (~$55)

In [ ]:
def build_credit_rows(project_id, version, leed_system, credit_mappings):
    rows = []
    for cm in credit_mappings:
        rows.append({
            'project_id': project_id,
            'source_version': version,
            'leed_system': leed_system,
            'source_credit_name': cm.get('credit_name', ''),
            'v5_credit_code': cm.get('v5_code', 'UNKNOWN'),
            'v5_category': cm.get('v5_category'),
            'points_awarded': cm.get('awarded', 0),
            'points_possible': cm.get('possible', 0),
            'mapping_method': 'rule' if cm.get('matched') else 'unmatched',
        })
    return rows

feature_rows, credit_rows, log_rows = [], [], []
counters = {'success': 0, 'failed': 0}

for i, pdf in enumerate(pdf_files, 1):
    if i % 50 == 0 or i == 1 or i == len(pdf_files):
        print(f'[{i:3d}/{len(pdf_files)}] {pdf.name[:55]}')
    try:
        state = run_standardization(pdf_path=str(pdf), directory_df=csv_df)
        if state.get('status') != 'completed':
            raise RuntimeError(f"status={state.get('status')}")

        final = state['final_v5_data']
        project = state.get('project', {})
        rule_result = state.get('rule_mapping_result', {})
        math_result = state.get('math_validation_result', {})
        val_result = state.get('validation_result', {}) or {}

        version = final.get('original_version', 'unknown')
        project_id = final.get('project_id', pdf.name)
        leed_system = final.get('leed_system', '')

        feature_rows.append({
            'project_id': project_id,
            'project_name': final.get('project_name', ''),
            'leed_system': leed_system,
            'building_type': final.get('building_type', ''),
            'gross_area_sqm': final.get('gross_area_sqm', 0),
            'original_version': version,
            'certification_level': final.get('certification_level', ''),
            'total_score_original': final.get('total_score_original', 0),
            'total_score_v5': final.get('total_score_v5', 0),
            'achievement_ratio_original': final.get('achievement_ratio_original', 0),
            'achievement_ratio_v5': final.get('achievement_ratio_v5', 0),
            'standardization_track': final.get('standardization_track', 'rule'),
            'drift': math_result.get('ratio_drift', 0),
            'credit_rule_hit_rate': rule_result.get('credit_rule_hit_rate', None),
            **{k: v for k, v in final.items() if k.startswith('ratio_')},
            **{k: v for k, v in final.items() if k.startswith('score_v5_')},
        })

        cm = rule_result.get('credit_mappings', [])
        if cm:
            credit_rows.extend(build_credit_rows(project_id, version, leed_system, cm))
        else:
            for cat in rule_result.get('mapped_categories', {}):
                credit_rows.append({
                    'project_id': project_id, 'source_version': version, 'leed_system': leed_system,
                    'source_credit_name': f'[category] {cat}',
                    'v5_credit_code': f'CAT_{cat}', 'v5_category': cat,
                    'points_awarded': project.get('categories', {}).get(cat, 0),
                    'points_possible': project.get('categories_possible', {}).get(cat, 0),
                    'mapping_method': 'category_proportional',
                })

        if val_result:
            log_rows.append({
                'project_id': project_id,
                'llm_review_target': val_result.get('target'),
                'llm_review_is_valid': val_result.get('is_valid'),
                'llm_review_score': val_result.get('validation_score'),
                'llm_review_issues': '; '.join(val_result.get('issues', []))[:300],
                'llm_review_feedback': (val_result.get('feedback') or '')[:300],
            })

        counters['success'] += 1
    except Exception:
        counters['failed'] += 1

print(f'\n완료: {counters}')

## 3단계: parquet 저장

In [ ]:
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

feat_df = pd.DataFrame(feature_rows)
cred_df = pd.DataFrame(credit_rows)

feat_df.to_parquet(PROCESSED / 'project_features.parquet', index=False)
cred_df.to_parquet(PROCESSED / 'standardized_credits.parquet', index=False)

if log_rows:
    log_df = pd.DataFrame(log_rows)
    log_df['project_id'] = log_df['project_id'].astype(str)
    feat_df['project_id'] = feat_df['project_id'].astype(str)
    merged = feat_df.merge(log_df, on='project_id', how='left')
    merged['has_llm_review'] = merged['llm_review_target'].notna()
    merged.to_parquet(PROCESSED / 'project_features_option_a.parquet', index=False)
    print(f'Option A parquet: {len(merged)}행 (LLM 리뷰 {merged["has_llm_review"].sum()}건)')

print(f'project_features.parquet: {len(feat_df)}행')
print(f'standardized_credits.parquet: {len(cred_df)}행')

## 4단계: 검증

In [ ]:
print('=== 기본 통계 ===')
print(f'행수: {len(feat_df)}')
print(f"버전: {feat_df['original_version'].value_counts().to_dict()}")
print(f"등급: {feat_df['certification_level'].value_counts().to_dict()}")
print(f"평균 drift: {feat_df['drift'].mean()*100:.2f}%")
print(f"drift > 20%: {(feat_df['drift'] > 0.20).sum()}건")
print(f"\n크레딧 매핑 방식:\n{cred_df['mapping_method'].value_counts()}")

## 다음

`02_데이터분석.ipynb` 에서 XGBoost + SHAP 실행.